In [1]:
import os
import pandas as pd
import datetime as dt
from urllib.parse import urlparse
import re
import numpy as np

### Part 1: Combine multiple tables into a single table


In [2]:
# --- Cấu hình đường dẫn (chỉnh ở đây nếu chạy trên máy khác) ---
RAW_DIR = "Raw Data"
CLEANED_DIR = "Cleaned_Data"

frames = []
for file in sorted(os.listdir(RAW_DIR)):
    if not file.endswith(".xlsx") or file.startswith("~$"):
        continue
    df_tmp = pd.read_excel(os.path.join(RAW_DIR, file))
    print(f"  {file}: {len(df_tmp):,} dòng")
    frames.append(df_tmp)

df = pd.concat(frames, ignore_index=True)
print(f"Sau concat         : {len(df):,} dòng")

# Chốt chặn: loại các bản ghi trùng khít toàn bộ cột
# (vd khi thư mục Raw Data lỡ chứa 2 lần cùng một file export)
before = len(df)
df = df.drop_duplicates(ignore_index=True)
print(f"Sau drop_duplicates: {len(df):,} dòng  (-{before - len(df):,} dòng trùng)")


  Socom_04e14a99bbb240a29b11ba42501e7b20.xlsx: 18,339 dòng
  Socom_0fcb53d5c8ea41a4bc91888407aad9a4.xlsx: 4,992 dòng
  Socom_c8310f3ad637482a906c628998c5bd38.xlsx: 126 dòng
Sau concat         : 23,457 dòng
Sau drop_duplicates: 23,457 dòng  (-0 dòng trùng)


In [3]:
df.head(10)

,Nhà sản xuất,Khách hàng,Email khách hàng,Năm,Tháng,Ngày,Nguồn lưu lượng (Traffic),Tham số,Nguồn giới thiệu (Referral),UTM_campaign,...,T.trạng đ.hàng,Phương thức thanh toán,Số đơn hàng,Doanh thu,Tiền khuyến mãi,Doanh thu thuần,Tổng hóa đơn,Đã thu,Số lượng,Vận chuyển
0,--,Hoàng khánh ngọc,khanhngoc20077@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,http://m.facebook.com,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,0,0.0,0.0,30000.0,0.0,0,30000
1,Maybelline New York,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,Direct,--,--,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,199000,0.0,199000.0,199000.0,0.0,1,0
2,--,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,Direct,--,--,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,0,0.0,0.0,30000.0,0.0,0,30000
3,L'Oréal Paris,Ms Tú,tultn05@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,210005,http://m.facebook.com,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,1,-1.0,0.0,0.0,0.0,1,0
4,L'Oréal Paris,Ms Tú,tultn05@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,210005,http://m.facebook.com,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,2,-2.0,0.0,0.0,0.0,2,0
5,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,199000,-20000.0,179000.0,179000.0,0.0,1,0
6,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,1,0.0,1.0,1.0,0.0,1,0
7,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,109000,0.0,109000.0,109000.0,0.0,1,0
8,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,169000,0.0,169000.0,169000.0,0.0,1,0
9,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,169000,0.0,169000.0,169000.0,0.0,1,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23457 entries, 0 to 23456
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Nhà sản xuất                 23457 non-null  object        
 1   Khách hàng                   23457 non-null  object        
 2   Email khách hàng             23457 non-null  object        
 3   Năm                          23457 non-null  datetime64[ns]
 4   Tháng                        23457 non-null  datetime64[ns]
 5   Ngày                         23457 non-null  datetime64[ns]
 6   Nguồn lưu lượng (Traffic)    23457 non-null  object        
 7   Tham số                      23227 non-null  object        
 8   Nguồn giới thiệu (Referral)  23457 non-null  object        
 9   UTM_campaign                 23457 non-null  object        
 10  UTM_content                  23457 non-null  object        
 11  UTM_medium                   23457 non-nu

##### 1. Change table column name from vietnamese to english


In [5]:
rename_columns = {
    "Nhà sản xuất": "manufacturer",
    "Khách hàng": "customer",
    "Email khách hàng": "customer_email",
    "Năm": "year",
    "Tháng": "month",
    "Ngày": "date",
    "Nguồn lưu lượng (Traffic)": "traffic_source",
    "Tham số": "parameter",
    "Nguồn giới thiệu (Referral)": "referral",
    "UTM_campaign": "utm_campaign",
    "UTM_content": "utm_content",
    "UTM_medium": "utm_medium",
    "UTM_source": "utm_source",
    "UTM_term": "utm_term",
    "Chi nhánh": "branch",
    "Nhân viên tạo": "created_by",
    "Loại sản phẩm": "product_category",
    "Tỉnh thành": "province",
    "Trang": "page",
    "Kênh bán hàng": "sale_channel",
    "Đơn hàng": "order_id",
    "Sản phẩm": "product_name",
    "SKU": "sku",
    "Quận huyện vận chuyển": "district",
    "Phiên bản": "version",
    "T.trạng t.toán": "payment_status",
    "T.trạng đ.hàng": "order_status",
    "Phương thức thanh toán": "payment_method",
    "Số đơn hàng": "order_count",
    "Doanh thu": "revenue",
    "Tiền khuyến mãi": "discount_amount",
    "Doanh thu thuần": "net_revenue",
    "Tổng hóa đơn": "total_invoice",
    "Đã thu": "amount_received",
    "Số lượng": "quantity",
    "Vận chuyển": "shipping_fee"
}

df.rename(columns=rename_columns, inplace=True)

In [6]:
df.head(10)

,manufacturer,customer,customer_email,year,month,date,traffic_source,parameter,referral,utm_campaign,...,order_status,payment_method,order_count,revenue,discount_amount,net_revenue,total_invoice,amount_received,quantity,shipping_fee
0,--,Hoàng khánh ngọc,khanhngoc20077@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,http://m.facebook.com,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,0,0.0,0.0,30000.0,0.0,0,30000
1,Maybelline New York,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,Direct,--,--,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,199000,0.0,199000.0,199000.0,0.0,1,0
2,--,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,Direct,--,--,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,0,0.0,0.0,30000.0,0.0,0,30000
3,L'Oréal Paris,Ms Tú,tultn05@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,210005,http://m.facebook.com,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,1,-1.0,0.0,0.0,0.0,1,0
4,L'Oréal Paris,Ms Tú,tultn05@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,210005,http://m.facebook.com,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,2,-2.0,0.0,0.0,0.0,2,0
5,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,199000,-20000.0,179000.0,179000.0,0.0,1,0
6,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,1,0.0,1.0,1.0,0.0,1,0
7,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,109000,0.0,109000.0,109000.0,0.0,1,0
8,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,169000,0.0,169000.0,169000.0,0.0,1,0
9,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-01-01,2021-05-01,2021-05-26,fb,--,https://shop.lorealparis.com.vn//?ref=210007&u...,--,...,Không hủy,Thanh toán khi giao hàng (COD),1,169000,0.0,169000.0,169000.0,0.0,1,0


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23457 entries, 0 to 23456
Data columns (total 36 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   manufacturer      23457 non-null  object        
 1   customer          23457 non-null  object        
 2   customer_email    23457 non-null  object        
 3   year              23457 non-null  datetime64[ns]
 4   month             23457 non-null  datetime64[ns]
 5   date              23457 non-null  datetime64[ns]
 6   traffic_source    23457 non-null  object        
 7   parameter         23227 non-null  object        
 8   referral          23457 non-null  object        
 9   utm_campaign      23457 non-null  object        
 10  utm_content       23457 non-null  object        
 11  utm_medium        23457 non-null  object        
 12  utm_source        23457 non-null  object        
 13  utm_term          23457 non-null  object        
 14  branch            2345

##### 2. Drop some unnecessary columns for transaction data


In [ ]:
# các cột year, month ở dạng datetime chỉ chứa dữ liệu về năm, tháng nên trở nên dư thừa
# các cột như parameter, utm_campaign, utm_content, utm_medium, utm_source, utm_term, created_by, page, sale_chanel, sku không có đầy dủ thông tin hoặc không có tác dụng cho mục đích xây báo cáo sau này
# cột payment_status không có tác dụng bởi vì đây là dữ liệu từ năm 2021

df.drop(['year', 'month', 'parameter', 'utm_campaign', 'utm_content', 'utm_medium', 'utm_source', 'utm_term', 'created_by', 'page', 'sale_channel', 'sku', 'payment_status', 'order_count', 'net_revenue'], axis=1, inplace=True, errors='ignore')

### Part 2: Data Cleaning


In [9]:
# Tách 3 luồng dữ liệu từ bảng gốc:
#   - shipping : product_name == '--'
#   - quà tặng : product_name chứa [quà tặng] / [gift] / [Quà tặng không bán]
#   - giao dịch: phần còn lại
GIFT_PATTERN = r'\[(?:quà tặng|gift|Quà tặng không bán)\]'

is_shipping = df['product_name'] == '--'
is_gift = df['product_name'].str.contains(GIFT_PATTERN, case=False, na=False)

shipping_df = df.loc[is_shipping, ['order_id', 'shipping_fee']].copy()
df_gift = df.loc[is_gift, ['order_id', 'product_name']].copy()
df = df.loc[~is_shipping & ~is_gift].copy()

print(f"shipping : {len(shipping_df):,} dòng")
print(f"quà tặng : {len(df_gift):,} dòng")
print(f"giao dịch: {len(df):,} dòng")

# Làm sạch tên quà tặng: bỏ phần tag [quà tặng] / [gift] / ...
df_gift = df_gift.rename(columns={'product_name': 'gift_name'})
df_gift['gift_name'] = (
    df_gift['gift_name'].str.replace(GIFT_PATTERN, '', case=False, regex=True).str.strip()
)

# date: datetime -> date (chỉ đổi khi còn là datetime, để chạy lại cell không lỗi)
if pd.api.types.is_datetime64_any_dtype(df['date']):
    df['date'] = df['date'].dt.date


shipping : 2,140 dòng
quà tặng : 2,185 dòng
giao dịch: 19,132 dòng


In [10]:
# Một số dữ liệu ở cột referral quá dài không phù hợp để dưa vào mysql
# Cần chuẩn hoá lại dữ liệu của cột mà vẫn giữ nguyên khả năng đưa ra insight

def extract_base_url(url):
    try:
        parsed = urlparse(str(url))
        return f"{parsed.scheme}://{parsed.netloc}/" if parsed.netloc else None
    except Exception:
        return None

if 'referral' in df.columns:
    df['clean_referral'] = df['referral'].apply(extract_base_url)

In [11]:
df[['referral', 'clean_referral']].drop_duplicates().head(10)

,referral,clean_referral
1,--,None
3,http://m.facebook.com,http://m.facebook.com/
5,https://shop.lorealparis.com.vn//?ref=210007&u...,https://shop.lorealparis.com.vn/
10,https://www.google.com/,https://www.google.com/
24,https://www.youtube.com/,https://www.youtube.com/
33,https://l.facebook.com/,https://l.facebook.com/
68,http://m.facebook.com/,http://m.facebook.com/
273,https://m.kenh14.vn/5-kem-chong-nang-hack-da-d...,https://m.kenh14.vn/
414,https://shop.maybelline.vn/collections/son-kem...,https://shop.maybelline.vn/
578,https://shop.maybelline.vn//?ref=210007&utm_so...,https://shop.maybelline.vn/


In [12]:
# Thay referral bằng base-url đã chuẩn hoá, rồi bỏ cột tạm
if 'clean_referral' in df.columns:
    df['referral'] = df['clean_referral']
    df.drop(columns=['clean_referral'], inplace=True)


In [13]:
df['referral'].value_counts()

referral
https://shop.lorealparis.com.vn/         1901
https://www.goshopback.vn/               1879
http://m.facebook.com/                   1811
https://l.facebook.com/                  1084
https://click.accesstrade.vn/            1031
https://www.google.com/                   581
https://shop.maybelline.vn/               554
https://www.lorealparis.com.vn/           393
https://l.instagram.com/                  293
https://instabio.cc/                       94
https://www.instagram.com/                 70
https://www.youtube.com/                   66
android-app://org.telegram.messenger/      40
http://linksanpham.clix9.com/              30
https://www.google.com.vn/                 29
https://m.facebook.com/                    28
https://fanlnk.to/                         18
https://pub.accesstrade.vn/                16
https://vinid.net/                         16
https://m.kenh14.vn/                       14
android-app://com.google.android.gm/       10
https://lm.facebook.com/ 

In [14]:
# Trích xuất tên kênh từ các url ở trên

# Danh sách từ khóa & kênh tương ứng
channel_map = {
    'facebook': 'facebook',
    'instagram': 'instagram',
    'youtube': 'youtube',
    'google': 'google',
    'kenh14': 'kenh14',
    'maybelline': 'maybelline',
    'lorealparis': 'lorealparis',
    'harasocial': 'harasocial',
    'accesstrade': 'accesstrade',
    'shopback': 'goshopback',
    'coccoc': 'coccoc',
    'telegram': 'telegram',
    'messenger': 'messenger',
    'vinid': 'vinid',
    'fanlnk': 'fanlnk',        # URL thực tế là fanlnk.to (không phải fanlink.to)
    'instabio': 'instabio',
    'linksanpham': 'linksanpham',
    'ecosia': 'ecosia'
}

def match_channel(url):
    if pd.isna(url):
        return 'none'
    url = str(url).lower()
    for keyword, channel in channel_map.items():
        if keyword in url:
            return channel
    return 'other'


df['Channel'] = df['referral'].apply(match_channel)


In [15]:
df['Channel'].value_counts()

Channel
none           9145
facebook       2933
lorealparis    2294
goshopback     1879
accesstrade    1049
google          620
maybelline      559
instagram       363
instabio         94
youtube          66
telegram         40
linksanpham      30
kenh14           18
fanlnk           18
vinid            16
messenger         3
coccoc            2
harasocial        2
ecosia            1
Name: count, dtype: int64

In [16]:
df['traffic_source'].value_counts()

traffic_source
Direct           9111
Referral         5924
fb               3749
Social            346
Search Engine       2
Name: count, dtype: int64

In [17]:
# Xử lý các cột liên quan đến traffice và url để phù hợp với việc phân tích

channel_to_source = {
    "none": "Direct",
    "facebook": "Facebook",
    "lorealparis": "Affiliate",
    "goshopback": "Affiliate",
    "accesstrade": "Affiliate",
    "google": "Search Engine",
    "maybelline": "Affiliate",
    "instagram": "Social",
    "instabio": "Social",
    "youtube": "Social",
    "telegram": "Social",
    "linksanpham": "Affiliate",
    "kenh14": "Affiliate",
    "other": "Affiliate",
    "vinid": "Affiliate",
    "harasocial": "Affiliate",
    "messenger": "Social",
    "coccoc": "Search Engine",
    "ecosia": "Search Engine",
    "fanlnk": "Affiliate"      # fanlnk.to là affiliate link
}

traffic_source_mapping = {
    "Direct": "Direct",
    "fb": "Facebook",
    "Referral": "Affiliate",
    "Social": "Social",
    "Search Engine": "Search Engine"
}

# Hàm xử lý logic chuyển đổi
def determine_final_source(row):
    ts = row['traffic_source']
    ch = row['Channel']
    
    if ts in traffic_source_mapping:
        return traffic_source_mapping[ts]
    elif ch in channel_to_source:
        return channel_to_source[ch]
    else:
        return "Affiliate"

# Tính traffic_source cuối cùng (chỉ chạy khi cột 'Channel' còn tồn tại -> chạy lại cell không lỗi)
if 'Channel' in df.columns:
    df['traffic_source'] = df.apply(determine_final_source, axis=1)
    df.drop(columns=['referral', 'Channel'], inplace=True, errors='ignore')

df['traffic_source'].value_counts()


traffic_source
Direct           9111
Affiliate        5924
Facebook         3749
Social            346
Search Engine       2
Name: count, dtype: int64

In [18]:
# Loại các bản ghi dùng email test của hệ thống (vd: test@test.com)
# Regex: bắt đầu bằng "test" HOẶC chứa "@test."
before = len(df)
df = df[~df['customer_email'].str.contains(r'(?:^test|@test\.)', regex=True, case=False, na=False)].copy()
print(f"Loại email test: -{before - len(df):,} dòng  |  còn {len(df):,} dòng")


Loại email test: -60 dòng  |  còn 19,072 dòng


In [19]:
# Đồng bộ gift/shipping với các order còn lại (sau khi đã loại email test)
# để tầng Bronze/Silver không phát sinh order_id mồ côi
valid_orders = set(df['order_id'].unique())
_bg, _bs = len(df_gift), len(shipping_df)
df_gift = df_gift[df_gift['order_id'].isin(valid_orders)].drop_duplicates(ignore_index=True)
shipping_df = shipping_df[shipping_df['order_id'].isin(valid_orders)].drop_duplicates(ignore_index=True)
print(f"gift    : {_bg:,} -> {len(df_gift):,}")
print(f"shipping: {_bs:,} -> {len(shipping_df):,}")


gift    : 2,185 -> 1,646
shipping: 2,140 -> 2,120


In [20]:
df.head(10)

,manufacturer,customer,customer_email,date,traffic_source,branch,product_category,province,order_id,product_name,district,version,order_status,payment_method,revenue,discount_amount,total_invoice,amount_received,quantity,shipping_fee
1,Maybelline New York,982285478,thuyan91096@gmail.com,2021-05-26,Direct,Kho Miền Bắc - MBL,Face,Cần Thơ,112403,Kem Nền Siêu Che Phủ Lâu Trôi Maybelline Super...,Quận Ninh Kiều,112 Natural Ivory,Không hủy,Thanh toán khi giao hàng (COD),199000,0.0,199000.0,0.0,1,0
3,L'Oréal Paris,Ms Tú,tultn05@gmail.com,2021-05-26,Facebook,Kho Miền Nam - OAP,Khác,Hồ Chí Minh,112402,Kem chống nắng L'Oreal Paris UV Perfect trắng ...,Quận 1,Default Title,Không hủy,Thanh toán khi giao hàng (COD),1,-1.0,0.0,0.0,1,0
4,L'Oréal Paris,Ms Tú,tultn05@gmail.com,2021-05-26,Facebook,Kho Miền Nam - OAP,Khác,Hồ Chí Minh,112402,Kem dưỡng L’Oreal Paris White Perfect ban ngày...,Quận 1,7ml,Không hủy,Thanh toán khi giao hàng (COD),2,-2.0,0.0,0.0,2,0
5,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-05-26,Facebook,Kho Miền Bắc - OAP,Khác,Thanh Hóa,112400,Kem chống nắng Nâng tông Giảm thâm Mịn Nhẹ Bảo...,Huyện Vĩnh Lộc,Kiềm dầu thoáng mịn (Viền Xanh lá),Không hủy,Thanh toán khi giao hàng (COD),199000,-20000.0,179000.0,0.0,1,0
6,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-05-26,Facebook,Kho Miền Bắc - OAP,Khác,Lai Châu,112399,Serum siêu cấp ẩm sáng da 7.5ml,Huyện Phong Thổ,Default Title,Không hủy,Thanh toán khi giao hàng (COD),1,0.0,1.0,0.0,1,0
7,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-05-26,Facebook,Kho Miền Bắc - OAP,Khác,Lai Châu,112399,Nước hoa hồng se khít lỗ chân lông và trắng mị...,Huyện Phong Thổ,200ml,Không hủy,Thanh toán khi giao hàng (COD),109000,0.0,109000.0,0.0,1,0
8,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-05-26,Facebook,Kho Miền Bắc - OAP,Dưỡng Chất,Lai Châu,112399,Kem dưỡng trắng sáng đều màu da L'Oreal White ...,Huyện Phong Thổ,50ml,Không hủy,Thanh toán khi giao hàng (COD),169000,0.0,169000.0,0.0,1,0
9,L'Oréal Paris,982285478,thuyan91096@gmail.com,2021-05-26,Facebook,Kho Miền Bắc - OAP,Dưỡng Chất,Lai Châu,112399,Kem dưỡng da trắng sáng L'Oreal White Perfect ...,Huyện Phong Thổ,50ml,Không hủy,Thanh toán khi giao hàng (COD),169000,0.0,169000.0,0.0,1,0
10,L'Oréal Paris,THÚY DIỄM TRẦN THỦY,hung.huynh@loreal.com,2021-05-26,Affiliate,Kho Miền Nam - OAP,Khác,Tây Ninh,112398,Kem chống nắng L'Oreal Paris UV Perfect trắng ...,Huyện Bến Cầu,Default Title,Không hủy,Thanh toán khi giao hàng (COD),4,0.0,4.0,0.0,4,0
11,L'Oréal Paris,THÚY DIỄM TRẦN THỦY,hung.huynh@loreal.com,2021-05-26,Affiliate,Kho Miền Nam - OAP,Sữa Rửa Mặt,Tây Ninh,112398,Sữa rửa mặt dưỡng trắng và sáng mịn da L'Oréal...,Huyện Bến Cầu,Default Title,Không hủy,Thanh toán khi giao hàng (COD),84000,0.0,84000.0,0.0,1,0


### Part 2b: Phân loại sản phẩm (category / sub_category)

1. Cell dưới xuất tên sản phẩm distinct ra `Cleaned_Data/product_names_distinct.csv`.
2. Chạy `python mapping_ollama.py` (Ollama) → sinh `Cleaned_Data/product_category_map.xlsx`.
3. Mở file `.xlsx`, sửa tay các dòng `UNKNOWN` / sai.
4. Chạy cell kế tiếp để map `category` + `sub_category` vào bảng giao dịch.


In [21]:
# Xuất danh sách tên sản phẩm distinct -> input cho mapping_ollama.py
import os
os.makedirs(CLEANED_DIR, exist_ok=True)
(df.groupby("product_name")
   .agg(n_rows=("product_name", "size"), n_orders=("order_id", "nunique"))
   .reset_index()
   .sort_values("n_rows", ascending=False)
   .to_csv(os.path.join(CLEANED_DIR, "product_names_distinct.csv"),
           index=False, encoding="utf-8-sig"))
print(f"{df['product_name'].nunique()} tên distinct -> {CLEANED_DIR}/product_names_distinct.csv")


146 tên distinct -> Cleaned_Data/product_names_distinct.csv


In [22]:
# Map product_category + sub_category từ file đã phân loại bằng AI + review tay:
#   Cleaned_Data/product_category_map.xlsx   (do mapping_ollama.py sinh ra)
import unicodedata

def _norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = s.replace("\u200b", "").replace("\u00a0", " ").replace("\u2019", "'").replace("\u2018", "'")
    return re.sub(r"\s+", " ", s).strip()

cat_map = pd.read_excel(os.path.join(CLEANED_DIR, "product_category_map.xlsx"))
cat_map["_key"] = cat_map["product_name"].map(_norm)
cat_map = cat_map.drop_duplicates("_key").set_index("_key")

_key = df["product_name"].map(_norm)
mapped_cat = _key.map(cat_map["category"])
mapped_sub = _key.map(cat_map["sub_category"])

missing = sorted(set(_key[mapped_cat.isna()]))
unknown = sorted(set(_key[mapped_cat.eq("UNKNOWN")]))
print(f"Không có trong map       : {len(missing)}")
for m in missing:
    print("   -", m)
print(f"Map = 'UNKNOWN' (cần sửa): {len(unknown)}")
for u in unknown:
    print("   -", u)

df["product_category"] = mapped_cat.fillna("UNKNOWN")
df["sub_category"] = mapped_sub.fillna("UNKNOWN")
df[["product_name", "product_category", "sub_category"]].drop_duplicates().head()


Không có trong map       : 0
Map = 'UNKNOWN' (cần sửa): 0


,product_name,product_category,sub_category
1,Kem Nền Siêu Che Phủ Lâu Trôi Maybelline Super...,Makeup,Face
3,Kem chống nắng L'Oreal Paris UV Perfect trắng ...,Skincare,Body Care
4,Kem dưỡng L’Oreal Paris White Perfect ban ngày...,Skincare,Lip Care
5,Kem chống nắng Nâng tông Giảm thâm Mịn Nhẹ Bảo...,Skincare,Body Care
6,Serum siêu cấp ẩm sáng da 7.5ml,Skincare,Face Care


### Part 3: Xuất dữ liệu đã làm sạch ra CSV


In [ ]:
# Xuất 3 file cho tầng Bronze.
# Thứ tự cột của Transaction khớp bảng bronze.Transaction_Data (21 cột, sub_category ở CUỐI).
os.makedirs(CLEANED_DIR, exist_ok=True)

BRONZE_TXN_COLS = [
    'manufacturer', 'customer', 'customer_email', 'date', 'traffic_source', 'branch',
    'product_category', 'province', 'order_id', 'product_name', 'district', 'version',
    'order_status', 'payment_method', 'revenue', 'discount_amount', 'total_invoice',
    'amount_received', 'quantity', 'shipping_fee', 'sub_category',
]
missing = [c for c in BRONZE_TXN_COLS if c not in df.columns]
assert not missing, f"Thiếu cột so với schema bronze: {missing}"

df[BRONZE_TXN_COLS].to_csv(
    os.path.join(CLEANED_DIR, 'Transaction_Data.csv'), index=False, encoding='utf-8-sig')
df_gift[['order_id', 'gift_name']].to_csv(
    os.path.join(CLEANED_DIR, 'Gift_Data.csv'), index=False, encoding='utf-8-sig')
shipping_df[['order_id', 'shipping_fee']].to_csv(
    os.path.join(CLEANED_DIR, 'Shipping_Data.csv'), index=False, encoding='utf-8-sig')

print("Đã xuất vào", CLEANED_DIR, "->", sorted(os.listdir(CLEANED_DIR)))
print(f"  Transaction_Data.csv: {len(df):,} dòng")
print(f"  Gift_Data.csv       : {len(df_gift):,} dòng")
print(f"  Shipping_Data.csv   : {len(shipping_df):,} dòng")